# Spatial Clustering Analysis: Ritel Modern Jakarta Selatan

Notebook ini merupakan kelanjutan dari analisis pemetaan ritel modern vs pasar tradisional.
Fokus utama pada tahap ini adalah:
1. **Load Dataset Final**
2. **K-Means Clustering**: Mempartisi minimarket ke dalam kluster terpusat.
3. **DBSCAN Clustering**: Mendeteksi *hotspot* (area over-saturasi) secara spasial berdasarkan kepadatan jarak.
4. **Visualisasi Map Interaktif**: Menggunakan Folium.
5. **Analisis Hotspot**: Evaluasi area dengan tingkat pelanggaran zonasi tertinggi.

In [20]:
# 1. IMPORT LIBRARY
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics.pairwise import haversine_distances
from math import radians

# Install folium jika belum ada
try:
    import folium
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'folium', '-q'])
    import folium

import warnings
warnings.filterwarnings('ignore')

In [21]:
# 2. LOAD DATASET FINAL
# Menggunakan path dari notebook sebelumnya
import subprocess

repo_url  = "https://github.com/shineistu86/CPS-CC26.git"
repo_name = "CPS-CC26"

# Clone repo jika belum ada
if not os.path.exists(repo_name):
    subprocess.run(["git", "clone", repo_url], check=True)

BASE_DIR_MAIN = os.path.join(os.getcwd(), repo_name, "data-mentah")
file_path = os.path.join(BASE_DIR_MAIN, "DATA_MINIMARKET_ZONASI_FINAL.csv")

try:
    df = pd.read_csv(file_path)
    print(f"Dataset berhasil dimuat dari: {file_path}")

except FileNotFoundError:
    # Fallback jika file berada di root direktori
    print("Path github tidak ditemukan, mencari file di repository...")

    found = False

    for root, dirs, files in os.walk(repo_name):
        if "DATA_MINIMARKET_ZONASI_FINAL.csv" in files:
            file_path = os.path.join(root, "DATA_MINIMARKET_ZONASI_FINAL.csv")
            df = pd.read_csv(file_path)
            print(f"Dataset berhasil dimuat dari: {file_path}")
            found = True
            break

    if not found:
        raise FileNotFoundError("DATA_MINIMARKET_ZONASI_FINAL.csv tidak ditemukan.")

# Pastikan tidak ada data koordinat yang kosong
df.dropna(subset=['latitude', 'longitude'], inplace=True)
print(f"Total baris data: {df.shape[0]}")
display(df[['nama_tempat', 'store', 'latitude', 'longitude', 'pelanggaran_<500m']].head())

Dataset berhasil dimuat dari: c:\Users\Devi\CPS-CC26\CPS-CC26\data-mentah\DATA_MINIMARKET_ZONASI_FINAL.csv
Total baris data: 662


,nama_tempat,store,latitude,longitude,pelanggaran_<500m
0,indomaret,Indomaret,-6.302203,106.791936,No
1,indomaret jeruk purut,Indomaret,-6.291165,106.812950,No
2,indomaret,Indomaret,-6.307003,106.793690,No
3,indomaret bdn raya,Indomaret,-6.279392,106.798442,No
4,indomaret,Indomaret,-6.278223,106.797096,No


In [22]:
# 3. K-MEANS CLUSTERING (Zonasi Wilayah)
coords = df[['latitude', 'longitude']].values

# Kita gunakan 10 kluster merepresentasikan pendekatan kecamatan di Jaksel
kmeans = KMeans(n_clusters=10, random_state=42, n_init=10)
df['kmeans_cluster'] = kmeans.fit_predict(coords)

print("Distribusi Minimarket per K-Means Cluster:")
display(df['kmeans_cluster'].value_counts().sort_index())

Distribusi Minimarket per K-Means Cluster:


kmeans_cluster
0    83
1    48
2    70
3    57
4    68
5    61
6    82
7    62
8    65
9    66
Name: count, dtype: int64

In [23]:
# 4. DBSCAN CLUSTERING (Deteksi Hotspot)
# Untuk mendeteksi kumpulan toko yang sangat padat (hotspot).
# Radius (eps) = 300 meter. Minimal 4 toko dalam radius tersebut untuk menjadi hotspot.

eps_meters = 300
earth_radius_meters = 6371000
eps_rad = eps_meters / earth_radius_meters

coords_rad = np.radians(coords)

dbscan = DBSCAN(eps=eps_rad, min_samples=4, algorithm='ball_tree', metric='haversine')
df['dbscan_cluster'] = dbscan.fit_predict(coords_rad)

n_hotspots = len(set(df['dbscan_cluster'])) - (1 if -1 in df['dbscan_cluster'] else 0)
n_noise = list(df['dbscan_cluster']).count(-1)

print(f"Ditemukan {n_hotspots} area Hotspot (kepadatan tinggi).")
print(f"Terdapat {n_noise} minimarket yang tidak masuk dalam hotspot (Noise / tersebar).")

Ditemukan 48 area Hotspot (kepadatan tinggi).
Terdapat 391 minimarket yang tidak masuk dalam hotspot (Noise / tersebar).


In [24]:
# 5. VISUALISASI CLUSTER (FOLIUM MAP)
# Titik tengah Jakarta Selatan
center_lat, center_lon = df['latitude'].mean(), df['longitude'].mean()
m = folium.Map(location=[center_lat, center_lon], zoom_start=13, tiles="CartoDB Positron")

# Palet warna untuk hotspot
colors = ['#e6194b', '#3cb44b', '#ffe119', '#4363d8', '#f58231',
          '#911eb4', '#46f0f0', '#f032e6', '#bcf60c', '#fabebe']

for idx, row in df.iterrows():
    cluster = row['dbscan_cluster']

    # Jika -1 (noise/bukan hotspot), warna abu-abu kecil
    if cluster == -1:
        color = '#cccccc'
        radius = 3
        fill_opacity = 0.4
    else:
        color = colors[cluster % len(colors)]
        radius = 7
        fill_opacity = 0.8

    popup_text = f"""
    <b>{row['store']}</b><br>
    Hotspot ID: {cluster if cluster != -1 else 'Bukan Hotspot'}<br>
    Pelanggaran Zonasi: {row['pelanggaran_<500m']}<br>
    Kecamatan: {row['nama_kecamatan']}
    """

    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=radius,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=fill_opacity,
        popup=folium.Popup(popup_text, max_width=250)
    ).add_to(m)

# Tampilkan map
m.save("minimarket_hotspots.html")

In [25]:
# 6. ANALISIS HOTSPOT & ZONASI MERAH
# Ambil data yang termasuk dalam kluster hotspot (mengabaikan noise -1)
hotspot_data = df[df['dbscan_cluster'] != -1]

hotspot_summary = hotspot_data.groupby('dbscan_cluster').agg(
    kecamatan_mayoritas=('nama_kecamatan', lambda x: x.mode()[0]),
    total_minimarket=('store', 'count'),
    indomaret_count=('store', lambda x: (x == 'Indomaret').sum()),
    alfamart_count=('store', lambda x: (x == 'Alfamart').sum()),
    total_pelanggaran=('pelanggaran_<500m', lambda x: (x == 'Yes').sum())
).reset_index()

# Hitung persentase pelanggaran per hotspot
hotspot_summary['pct_pelanggaran'] = (hotspot_summary['total_pelanggaran'] / hotspot_summary['total_minimarket'] * 100).round(1)

# Urutkan berdasarkan hotspot dengan pelanggaran terbanyak
hotspot_summary = hotspot_summary.sort_values(by='total_pelanggaran', ascending=False).reset_index(drop=True)

print("RINGKASAN HOTSPOT MINIMARKET (Area Tersaturasi Secara Spasial)")
display(hotspot_summary)

print("\nKesimpulan Hotspot:")
top_hotspot = hotspot_summary.iloc[0]
print(f"Hotspot dengan risiko tertinggi adalah Cluster {top_hotspot['dbscan_cluster']} di dominasi area {top_hotspot['kecamatan_mayoritas']}.")
print(f"Terdapat {top_hotspot['total_minimarket']} minimarket menumpuk di radius yang berdekatan, dengan tingkat pelanggaran zonasi sebesar {top_hotspot['pct_pelanggaran']}%.")

RINGKASAN HOTSPOT MINIMARKET (Area Tersaturasi Secara Spasial)


,dbscan_cluster,kecamatan_mayoritas,total_minimarket,indomaret_count,alfamart_count,total_pelanggaran,pct_pelanggaran
0,23,Mampang Prapatan,8,3,5,7,87.5
1,9,Jagakarsa,6,3,3,6,100.0
2,26,Pesanggrahan,5,2,3,5,100.0
3,20,Kebayoran Baru,4,3,1,4,100.0
4,17,Mampang Prapatan,4,1,3,4,100.0
5,25,Setiabudi,5,2,3,4,80.0
6,27,Kebayoran Lama,4,2,2,4,100.0
7,46,Tebet,4,0,4,3,75.0
8,33,Pancoran,4,2,2,2,50.0
9,2,Cilandak,8,6,2,2,25.0



Kesimpulan Hotspot:
Hotspot dengan risiko tertinggi adalah Cluster 23 di dominasi area Mampang Prapatan.
Terdapat 8 minimarket menumpuk di radius yang berdekatan, dengan tingkat pelanggaran zonasi sebesar 87.5%.


In [26]:
df['kmeans_cluster'].value_counts()

kmeans_cluster
0    83
6    82
2    70
4    68
9    66
8    65
7    62
5    61
3    57
1    48
Name: count, dtype: int64

In [27]:
df_2 = pd.read_csv('Data Clean/jaksel_spatial_features_v4_AI_ready.csv')
df_2

,nama_tempat,rating_tempat,user_ratings_total,latitude,longitude,alamat_tempat,place_id,store,nama_kelurahan,nama_kecamatan,nama_kota,competitor_density_500m,jarak_kompetitor_meter,kompetitor_head_to_head,jarak_pasar_meter,pasar_terdekat,pelanggaran_zonasi,cluster_kmeans_makro,cluster_dbscan_hotspot
0,indomaret,0.0,0,-6.214337,106.831213,"wisma budi, jl. h. r. rasuna said no.3, rt.3/r...",ChIJVwsQapH1aS4RRn0qO3oi72c,Indomaret,Karet,Setiabudi,Kota Jakarta Selatan,5,363.51,0,545.823150,pasar mencos,0,9,-1
1,indomaret dr. saharjo,4.2,16,-6.216004,106.844071,"jl. dr. saharjo no.109, rt.2/rw.10, manggarai ...",ChIJk___P4nzaS4R-mx5t36xeEE,Indomaret,Manggarai Selatan,Tebet,Kota Jakarta Selatan,10,134.85,0,1196.547259,pasar jaya bukit duri,0,4,0
2,indomaret soepomo,4.1,16,-6.232502,106.844433,"qr9v+2q4, jalan dr. soepomo, rt.1/rw.3, menten...",ChIJw4Qe95TzaS4R4TukR0p9hZk,Indomaret,Menteng Dalam,Tebet,Kota Jakarta Selatan,2,418.57,0,1405.849784,pasar pspt,0,4,-1
3,indomaret tebet,4.7,26,-6.224200,106.852534,"jl. kh abdullah syafei no.2, rt.12/rw.5, bukit...",ChIJndinnI_zaS4RsBdPM7-_7mo,Indomaret,Bukit Duri,Tebet,Kota Jakarta Selatan,6,297.81,0,721.506358,pasar jaya bukit duri,0,4,-1
4,indomaret menteng wadas,3.0,2,-6.210621,106.841945,"jl. menteng wadas timur no.17, rt.1/rw.7, ps. ...",ChIJVxLzin71aS4R5ngDmYA-lEM,Indomaret,Pasar Manggis,Setiabudi,Kota Jakarta Selatan,10,43.61,1,1634.960230,pasar jaya bukit duri,0,4,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
657,alfamart,0.0,0,-6.310546,106.793753,"jl. pd. labu raya no.7, rw.1, pd. labu, kota j...",ChIJmcOA6EXvaS4RQqMPIIRVwRA,Alfamart,Pondok Labu,Cilandak,Kota Jakarta Selatan,7,56.15,0,188.565085,pasar pondok labu,1,8,-1
658,alfamart,4.3,3,-6.288993,106.773820,"pq6f+cg4, ruko pasar jumat, jl. nasional 12, r...",ChIJQ8QJ52vxaS4R9ByNQALudjw,Alfamart,Pondok Pinang,Kebayoran Lama,Kota Jakarta Selatan,3,483.04,0,2384.966838,pasar mede pd pasar jaya,0,8,-1
659,alfamart,0.0,0,-6.279508,106.807225,"prc4+5ww, rt.6/rw.11, west cilandak, kota jaka...",ChIJg1z4LdjxaS4RtLCmQRpbnE0,Alfamart,Cilandak Barat,Cilandak,Kota Jakarta Selatan,0,970.84,0,150.093503,pasar cipete selatan,1,8,-1
660,alfamart margasatwa raya,4.6,54,-6.313074,106.800415,"jl. margasatwa no.90, rt.8/rw.2, pd. labu, kot...",ChIJ46jSRZvvaS4ReMMdsz0r-Ls,Alfamart,Pondok Labu,Cilandak,Kota Jakarta Selatan,5,38.57,1,741.675103,pasar pondok labu,0,8,-1


In [28]:
df_2.columns.to_list()

['nama_tempat',
 'rating_tempat',
 'user_ratings_total',
 'latitude',
 'longitude',
 'alamat_tempat',
 'place_id',
 'store',
 'nama_kelurahan',
 'nama_kecamatan',
 'nama_kota',
 'competitor_density_500m',
 'jarak_kompetitor_meter',
 'kompetitor_head_to_head',
 'jarak_pasar_meter',
 'pasar_terdekat',
 'pelanggaran_zonasi',
 'cluster_kmeans_makro',
 'cluster_dbscan_hotspot']

In [29]:
df_2['cluster_kmeans_makro'].value_counts()

cluster_kmeans_makro
3    87
4    79
2    73
5    70
1    68
8    66
6    63
0    63
9    47
7    46
Name: count, dtype: int64

In [30]:
df

,nama_tempat,rating_tempat,user_ratings_total,latitude,longitude,alamat_tempat,place_id,store,nama_kelurahan,nama_kecamatan,nama_kota,jarak_pasar_meter,pasar_terdekat,pelanggaran_<500m,kmeans_cluster,dbscan_cluster
0,indomaret,4.3,104,-6.302203,106.791936,"jl. lb. bulus iii no.40, rt.9/rw.7, cilandak b...",ChIJaxj8TyDuaS4RmJ8rDkpB2GA,Indomaret,Cilandak Barat,Cilandak,Kota Jakarta Selatan,1134.281500,pasar pondok labu,No,8,-1
1,indomaret jeruk purut,4.5,132,-6.291165,106.812950,"jl. jeruk purut no.22, rt.3/rw.3, cilandak tim...",ChIJvy_4mvnxaS4RjjbhIFaTp4Y,Indomaret,Cilandak Timur,Pasar Minggu,Kota Jakarta Selatan,1518.403739,pasar cipete selatan,No,8,-1
2,indomaret,0.0,0,-6.307003,106.793690,"10, rt.4/rw.10, pondok labu, south jakarta city",ChIJPz4W5BPvaS4RoUnKBAiaL3o,Indomaret,Pondok Labu,Cilandak,Kota Jakarta Selatan,582.576272,pasar pondok labu,No,8,-1
3,indomaret bdn raya,4.2,66,-6.279392,106.798442,"jl. bdn raya no.10, rt.10/rw.11, cilandak bar....",ChIJKZqyypTxaS4RkjwUtyXrkBY,Indomaret,Cilandak Barat,Cilandak,Kota Jakarta Selatan,891.544172,pasar mede pd pasar jaya,No,8,0
4,indomaret,4.7,3,-6.278223,106.797096,"jl. rs. fatmawati raya no.7, rt.8/rw.6, gandar...",ChIJzVqDlcTxaS4Rwlnqrz5hXbo,Indomaret,Gandaria Selatan,Cilandak,Kota Jakarta Selatan,971.213547,pasar mede pd pasar jaya,No,8,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
657,alfamart komplek selmis,0.0,0,-6.225834,106.858991,"jl. asem baris raya no.3, rt.3/rw.9, kebon bar...",ChIJ0dPodPrzaS4RE47Rr8Mz77Y,Alfamart,Kebon Baru,Tebet,Kota Jakarta Selatan,984.677250,pasar jaya bukit duri,No,9,31
658,alfamart bukit duri selatan,0.0,0,-6.221214,106.856739,"jl. cucakrawa no.882, rt.8/rw.4, bukit duri, k...",ChIJ8a2fcwfzaS4Rte3SBP5Gj_0,Alfamart,Bukit Duri,Tebet,Kota Jakarta Selatan,415.714786,pasar jaya bukit duri,Yes,9,46
659,alfamart stasiu,5.0,3,-6.225869,106.858647,"qvf5+mfc, flyover stasiun tebet, rt.3/rw.9, kb...",ChIJYRBiI-rzaS4RBW5LKzm2-yY,Alfamart,Kebon Baru,Tebet,Kota Jakarta Selatan,970.384349,pasar jaya bukit duri,No,9,31
660,alfamart barkah ii,0.0,0,-6.219769,106.851183,"jl. barkah i no.14, rw.6, manggarai sel., kota...",ChIJ85lVEnTzaS4RHy8f_3cmsuI,Alfamart,Manggarai Selatan,Tebet,Kota Jakarta Selatan,431.201690,pasar jaya bukit duri,Yes,9,-1


In [31]:
# Cek jumlah data tiap cluster
print(df_2['cluster_kmeans_makro'].value_counts())

# Menampilkan seluruh data pada masing-masing cluster
for c in sorted(df_2['cluster_kmeans_makro'].unique()):
    print("\n" + "="*60)
    print(f"CLUSTER {c}")
    print("="*60)
    
    cluster_data = df_2[df_2['cluster_kmeans_makro'] == c]
    
    print(cluster_data.head())   # tampilkan beberapa data awal
    print(f"\nJumlah data: {len(cluster_data)}")



# Pilih kolom numerik yang ingin dianalisis
kolom_numerik = [
    'competitor_density_500m',
    'jarak_kompetitor_meter',
    'kompetitor_head_to_head',
    'jarak_pasar_meter'
]

# Statistik deskriptif tiap cluster
for c in sorted(df_2['cluster_kmeans_makro'].unique()):
    print("\n" + "#"*70)
    print(f"STATISTIK CLUSTER {c}")
    print("#"*70)
    
    cluster_data = df_2[df_2['cluster_kmeans_makro'] == c]
    
    print(cluster_data[kolom_numerik].describe())


# =========================================================
# RATA-RATA TIAP CLUSTER
# =========================================================

# Untuk interpretasi lebih mudah
cluster_summary = (
    df_2.groupby('cluster_kmeans_makro')[kolom_numerik]
    .mean()
    .round(2)
)

print("\nRATA-RATA TIAP CLUSTER")
print(cluster_summary)


# MELIHAT KECAMATAN YANG MASUK TIAP CLUSTER
for c in sorted(df_2['cluster_kmeans_makro'].unique()):
    print("\n" + "="*50)
    print(f"KECAMATAN PADA CLUSTER {c}")
    print("="*50)
    
    kecamatan = df_2[df_2['cluster_kmeans_makro'] == c]['nama_kecamatan'].tolist()
    
    for k in kecamatan:
        print("-", k)


# Menyimpan masing-masing cluster ke file CSV
for c in sorted(df_2['cluster_kmeans_makro'].unique()):
    cluster_data = df_2[df_2['cluster_kmeans_makro'] == c]
    cluster_data.to_csv(
        f'cluster_{c}.csv',
        index=False
    )

print("\nFile cluster berhasil disimpan.")

cluster_kmeans_makro
3    87
4    79
2    73
5    70
1    68
8    66
6    63
0    63
9    47
7    46
Name: count, dtype: int64

CLUSTER 0
                       nama_tempat  rating_tempat  user_ratings_total  \
51    indomaret tegal parang. t986            4.1                   9   
52  indomaret tegal parang 2. ts86            3.9                  33   
54       indomaret mampang 8. t5wi            3.9                  17   
59                       indomaret            2.3                   3   
65    indomaret ciputat raya 2a -b            0.0                   0   

    latitude   longitude                                      alamat_tempat  \
51 -6.246465  106.832222  qr3j+cw7, jl. tegal parang, rt.7/rw.5, tegal p...   
52 -6.244198  106.832425  jl. tegal parang utara no.35, rt.7/rw.4, mampa...   
54 -6.247720  106.830637  jalan mampang prapatan viii, rt.03 / rw.02, te...   
59 -6.248328  106.824032  jl. mampang prapatan vii no.1, rw.3, tegal par...   
65 -6.239547  106.826350  jl

---
## Interpretasi Cluster
---

## Cluster 0

Cluster 0 Sebagian besar berada di kecamatan Mampang Prapatan dan beberapa di Pancoran, Setiabudi, Kebayaroran Baru, dan Pasar Minggu.

## Cluster 1

Cluster 1 hampir tersebar merata di kecamatan Pasar Minggu dan Jagakarsa.

## Cluster 2

Clsuter 2 tersebar di 2 kecamatan, yaitu Kebayoran Lama dan Pesanggrahan.

## Cluster 3

Cluster 3 tersebar di 2 kecamatan Kebayoran Lama dan Cilandak dengan sebagian besar berada di kecamatan Kebayoran Lama.

## Cluster 4

Cluster 4 didominasi oleh kecamatan Tebet dan diikuti oleh kecamatan Setiabudi. 

## Cluster 5
Cluster 5 berada di satu wilayah saja, yaitu kecamatan Jagakarsa.

## Cluster 6

Cluster 6 tersebar di kecamatan Pancoran dan Pasar Minggu dengan dominasi berada di Pancoran.

## Cluster 7

Cluster 7 cukup tersebar di 2 kecamatan, yaitu Kebayoran Lama dan Pesanggrahan.

## Cluster 8

Cluster 8 didominasi oleh kecamatan Cilandak dan diikuti oleh Kebayoran Lama dan Pasar Minggu.

## Cluster 9

Cluster 9 sebagian besar berada di kecamatan Setiabudi diikuti oleh Kabayoran Lama dan Mampang Prapatan. 